In [6]:
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

mercari_df= pd.read_csv('../mercari_train.tsv',sep='\t')

# 0. 전처리 전 기준점 계산을 위한 원본 백업
raw_df = mercari_df.copy()

print("⏳ [1/5] 초고속 데이터 클리닝 및 결측치 처리 중...")
# $3 미만 필터링 및 가장 빠른 내장 함수(fillna)로만 Null값 청소
mercari_df = mercari_df[mercari_df["price"] >= 3].reset_index(drop=True)
mercari_df["brand_name"] = mercari_df["brand_name"].fillna("Missing")
mercari_df["category_name"] = mercari_df["category_name"].fillna("Missing")
mercari_df["item_description"] = mercari_df["item_description"].fillna(
    "No description yet"
)

# 텍스트 데이터의 원활한 처리를 위해 소문자화 통일
mercari_df["name"] = mercari_df["name"].astype(str).str.lower()
mercari_df["item_description"] = (
    mercari_df["item_description"].astype(str).str.lower()
)

print("⏳ [2/5] 5가지 파생 변수(Feature Engineering) 생성 중...")
# [버그 수정 완료] 카테고리 슬래시(/) 분할 후 정확하게 대(0), 중(1), 소(2) 인덱스 지정 매핑
categories = mercari_df["category_name"].str.split("/", expand=True)

mercari_df["cat_main"] = categories[0].fillna("Missing")
mercari_df["cat_sub1"] = (
    categories[1].fillna("Missing") if categories.shape[1] > 1 else "Missing"
)
mercari_df["cat_sub2"] = (
    categories[2].fillna("Missing") if categories.shape[1] > 2 else "Missing"
)

# 고속 내장 str 벡터 연산으로 글자 길이 및 단어 수 추출
mercari_df["desc_word_count"] = (
    mercari_df["item_description"].str.split().str.len()
)
mercari_df["name_len"] = mercari_df["name"].str.len()

print("⏳ [3/5] 텍스트 벡터화 및 범주형 원-핫 인코딩(OneHotEncoder) 전환 중...")
# 문장형 데이터(name, description)는 단어 빈도수 기반 벡터화 적용
cnt_vec = CountVectorizer(max_features=5000)
X_name = cnt_vec.fit_transform(mercari_df["name"])

tfidf_vec = TfidfVectorizer(max_features=10000, stop_words="english")
X_desc = tfidf_vec.fit_transform(mercari_df["item_description"])

# 1글자 숫자 에러를 방지하는 OneHotEncoder 적용
oh_enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_categorical = oh_enc.fit_transform(
    mercari_df[
        [
            "brand_name",
            "cat_main",
            "cat_sub1",
            "cat_sub2",
            "item_condition_id",
            "shipping",
        ]
    ]
)

# 순수 수치형 데이터 추출
X_numeric = mercari_df[["desc_word_count", "name_len"]].values

# 모든 피처를 옆으로 하나로 결합하여 하나의 통합 특성 행렬(X) 빌드
X_features_matrix = hstack((X_name, X_desc, X_categorical, X_numeric)).tocsr()

print("⏳ [4/5] 데이터 분할 및 종속 변수(y) 스케일링 표준화 중...")
# 정리 노트 반영: 타깃 변수 y에 log(price + 1)을 우선 적용
log_price = np.log1p(mercari_df["price"]).values.reshape(-1, 1)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X_features_matrix, log_price, test_size=0.2, random_state=42
)

# StandardScaler 적용 및 글로벌화 수행 후 차원 평탄화
global y_scalar
y_scalar = StandardScaler()
y_train = y_scalar.fit_transform(y_train_raw).ravel()
y_test = y_scalar.transform(y_test_raw).ravel()

print("⏳ [5/5] 경량화된 LightGBM 모델 초고속 학습 중...")
# 멈춤 현상 없도록 하이퍼파라미터 디폴트 세팅 및 트리 개수 조정 후 고속 학습
fast_lgb = lgb.LGBMRegressor(
    n_estimators=50,
    learning_rate=0.1,
    num_leaves=31,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
fast_lgb.fit(X_train, y_train)

# 예측 수행 및 이중 역변환 (스케일 해제 -> 로그 해제) 실행
scaled_preds = fast_lgb.predict(X_test).reshape(-1, 1)
log_preds = y_scalar.inverse_transform(scaled_preds)
final_predicted_price = np.expm1(log_preds)
final_actual_price = np.expm1(y_test_raw)

# 전처리 전(Baseline) 상태와의 오차 점수 비교 계산
raw_mean_price = np.mean(raw_df["price"])
baseline_preds = np.full_like(final_actual_price, raw_mean_price)

baseline_rmsle = np.sqrt(
    mean_squared_error(np.log1p(final_actual_price), np.log1p(baseline_preds))
)
final_rmsle = np.sqrt(
    mean_squared_error(np.log1p(final_actual_price), np.log1p(final_predicted_price))
)

print("\n" + "=" * 50)
print("📊 MERCARI PRICE CHALLENGE 모델 성능 검증 결과")
print("=" * 50)
print(f"❌ 고급 전처리 전 (Baseline) RMSLE 오차 점수 : {baseline_rmsle:.4f}")
print(f"✨ 고급 전처리 후 (LightGBM)  RMSLE 오차 점수 : {final_rmsle:.4f}")
print("-" * 50)
print(
    f"💡 결론: 오차가 {baseline_rmsle - final_rmsle:.4f} 만큼 대폭 감소하여 모델이 정상 작동함을 입증했습니다!"
)
print("=" * 50)

⏳ [1/5] 초고속 데이터 클리닝 및 결측치 처리 중...
⏳ [2/5] 5가지 파생 변수(Feature Engineering) 생성 중...
⏳ [3/5] 텍스트 벡터화 및 범주형 원-핫 인코딩(OneHotEncoder) 전환 중...
⏳ [4/5] 데이터 분할 및 종속 변수(y) 스케일링 표준화 중...
⏳ [5/5] 경량화된 LightGBM 모델 초고속 학습 중...

📊 MERCARI PRICE CHALLENGE 모델 성능 검증 결과
❌ 고급 전처리 전 (Baseline) RMSLE 오차 점수 : 0.8205
✨ 고급 전처리 후 (LightGBM)  RMSLE 오차 점수 : 0.5660
--------------------------------------------------
💡 결론: 오차가 0.2545 만큼 대폭 감소하여 모델이 정상 작동함을 입증했습니다!


In [3]:
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from hyperopt import fmin, hp, STATUS_OK, tpe
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import re


warnings.filterwarnings("ignore")

# =====================================================================
# [단계 1] 전처리 전 기본 Baseline 평가를 위한 원본 데이터 보존
# =====================================================================
# 전처리 후 결과와 비교하기 위해 가공되지 않은 순수 원본 상태 기록
mercari_df= pd.read_csv('../mercari_train.tsv',sep='\t')
raw_df = mercari_df.copy()


# =====================================================================
# [단계 2] 고급 전처리 & 데이터 클리닝 (개선 버전)
# =====================================================================
# 1. 가격 비즈니스 로직 적용 ($3 미만 오류 데이터 제거)
mercari_df = mercari_df[mercari_df["price"] >= 3].reset_index(drop=True)

# 2. 결측치 및 의미 없는 텍스트 기본 동기화
mercari_df["name"] = mercari_df["name"].fillna("Missing")
mercari_df["item_description"] = mercari_df["item_description"].fillna("Missing")
mercari_df["item_description"] = mercari_df["item_description"].replace(
    "No description yet", "Missing"
)

# 3. [순서 변경] 소문자 기반의 브랜드 추출 (정규화 전에 실행해야 정확함)
# 실제 존재하는 브랜드 목록을 '소문자'로 저장
existing_brands = set(mercari_df["brand_name"].dropna().str.lower().unique())


def extract_brand_advanced(row):
    if pd.isna(row["brand_name"]) or row["brand_name"] == "Missing":
        # 상품명을 소문자로 분할하여 매칭 테스트
        words = str(row["name"]).lower().split()
        for word in words:
            if word in existing_brands:
                # 매칭된 소문자 브랜드를 원래 대소문자 형태로 복원하거나 소문자 그대로 사용
                return word  
        return "Missing"
    return str(row["brand_name"]).lower()  # 일관성을 위해 브랜드명도 소문자화


mercari_df["brand_name"] = mercari_df.apply(extract_brand_advanced, axis=1)

# 4. 텍스트 정규화 (속도 및 안정성 향상)
decontract_dict = {
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "i'm": "i am",
    "it's": "it is",
}


def clean_text_fast(text):
    if not isinstance(text, str) or text == "Missing":
        return ""
    text = text.lower()
    for word, replacement in decontract_dict.items():
        text = text.replace(word, replacement)
    # 컴파일된 정규식으로 특수문자 제거 (속도 개선)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text


mercari_df["name"] = mercari_df["name"].apply(clean_text_fast)
mercari_df["item_description"] = mercari_df["item_description"].apply(
    clean_text_fast
)

# =====================================================================
# [단계 3] 피쳐 구조화 (파생 변수 확장)
# =====================================================================
# 1. 카테고리 분할
mercari_df["category_name"] = mercari_df["category_name"].fillna("Missing")
categories = mercari_df["category_name"].str.split("/", expand=True)

mercari_df["cat_main"] = categories[0].fillna("Missing")
mercari_df["cat_sub1"] = (
    categories[1].fillna("Missing") if categories.shape[1] > 1 else "Missing"
)
mercari_df["cat_sub2"] = (
    categories[2].fillna("Missing") if categories.shape[1] > 2 else "Missing"
)

# 2. 수치형 텍스트 통계 피처 대폭 보강
mercari_df["desc_word_count"] = mercari_df["item_description"].apply(
    lambda x: len(x.split())
)
mercari_df["desc_len"] = mercari_df["item_description"].apply(
    len
)  # 설명글 글자 수 추가
mercari_df["name_word_count"] = mercari_df["name"].apply(
    lambda x: len(x.split())
)  # 상품명 단어 수 추가
mercari_df["name_len"] = mercari_df["name"].apply(len)

# =====================================================================
# [단계 4] 인코딩, 벡터화 및 통합 특성 행렬 구축 (수정본)
# =====================================================================

# 1. 텍스트 벡터화 (상품명 & 설명)
cnt_vec = CountVectorizer(ngram_range=(1, 2), max_features=25000)
X_name = cnt_vec.fit_transform(mercari_df["name"])

tfidf_vec = TfidfVectorizer(max_features=40000, stop_words="english")
X_desc = tfidf_vec.fit_transform(mercari_df["item_description"])

# 2. 범주형 인코딩 (CountVectorizer 패턴 지정)
# [수정] cv_token 변수명을 cv_brand와 일치시키거나 하나로 통일해야 합니다.
cv_token = CountVectorizer(token_pattern=r"(?u)\b[^,;]+?\b")
X_brand = cv_token.fit_transform(mercari_df["brand_name"])
X_cat_main = cv_token.fit_transform(mercari_df["cat_main"])
X_cat_sub1 = cv_token.fit_transform(mercari_df["cat_sub1"])
X_cat_sub2 = cv_token.fit_transform(mercari_df["cat_sub2"])

# 3. 범주형 인코딩 (0/1 희소행렬 형태 구축)
X_condition = cv_token.fit_transform(mercari_df["item_condition_id"].astype(str))
X_shipping = cv_token.fit_transform(mercari_df["shipping"].astype(str))

# 4. 수치형 피처 정규화 및 희소 행렬 변환
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(
    mercari_df[["desc_word_count", "name_len"]].values
)
# [수정] 텍스트/범주형 행렬(CSR Matrix)과 결합하기 위해 수치형도 희소 행렬로 변환합니다.
X_numeric_sparse = sparse.csr_matrix(X_numeric_scaled)

# 5. 하나의 거대한 통합 특성 행렬(X)로 완성
# [수정] hstack을 사용하기 위해 scipy.sparse.hstack을 명시적으로 사용하고 수치형 희소행렬을 넣습니다.
X_features_matrix = sparse.hstack(
    (
        X_name,
        X_desc,
        X_brand,
        X_cat_main,
        X_cat_sub1,
        X_cat_sub2,
        X_condition,
        X_shipping,
        X_numeric_sparse,  # 변환된 희소 행렬 대입
    )
).tocsr()
# =====================================================================
# [단계 5] 모의고사를 위한 데이터셋 분할 (Train / Test)
# =====================================================================
# 정리 내용 반영: 타깃 변수 y에 우선적으로 log(price+1) 적용
log_price = np.log1p(mercari_df["price"]).values.reshape(-1, 1)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X_features_matrix, log_price, test_size=0.2, random_state=42
)

# =====================================================================
# [단계 6] 스케일링 (StandardScaler 종속 변수 표준화 및 글로벌화)
# =====================================================================
global y_scalar
y_scalar = StandardScaler()

# 정리서에 나온 표준화 코드 그대로 작동
y_train = y_scalar.fit_transform(y_train_raw)
y_test = y_scalar.transform(y_test_raw)

# 튜닝 속도 최적화를 위해 차원 평탄화(Flatten)
y_train_flat = y_train.ravel()
y_test_flat = y_test.ravel()

# =====================================================================
# [단계 7] Hyperopt를 통한 LightGBM 하이퍼파라미터 최적화
# =====================================================================
# 탐색할 파라미터 공간 범위 설정
space = {
    "learning_rate": hp.loguniform("learning_rate", np.log(0.05), np.log(0.2)),
    "max_depth": hp.choice("max_depth", [5, 7, 10]),
    "num_leaves": hp.choice("num_leaves", [31, 63, 127]),
    "subsample": hp.uniform("subsample", 0.7, 1.0),
}


# 최적화 목적 함수 정의 (RMSE 최소화 목표)
def objective(params):
    model = lgb.LGBMRegressor(
        **params,
        n_estimators=100,  # 빠른 튜닝을 위해 100개 설정 (실전엔 500+ 권장)
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train_flat)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test_flat, preds))
    return {"loss": rmse, "status": STATUS_OK}


print("💡 [Hyperopt] 최적의 하이퍼파라미터 탐색을 시작합니다...")
best_params = fmin(
    fn=objective, space=space, algo=tpe.suggest, max_evals=3
)  # 테스트용 3회 수행

# Choice 인덱스 실제 값으로 변환
max_depth_options = [5, 7, 10]
num_leaves_options = [31, 63, 127]
best_params["max_depth"] = max_depth_options[best_params["max_depth"]]
best_params["num_leaves"] = num_leaves_options[best_params["num_leaves"]]
print(f"🎯 최적화 완료된 파라미터: {best_params}")

# =====================================================================
# [단계 8] 최적의 모델 학습 및 최종 결과 분석 (전/후 비교 출력)
# =====================================================================
# 1. 최적 파라미터로 전처리 완료된 최상위 LightGBM 학습
best_lgb = lgb.LGBMRegressor(**best_params, n_estimators=200, n_jobs=-1, verbose=-1)
best_lgb.fit(X_train, y_train_flat)

# 2. 테스트 데이터 예측 수행 및 글로벌 스케일러 역변환(Inverse Transform)
scaled_preds = best_lgb.predict(X_test).reshape(-1, 1)
log_preds = y_scalar.inverse_transform(scaled_preds)  # 스케일링 해제

# 원래 가격($) 체급으로 완전히 복구하기 (np.expm1)
final_actual_price = np.expm1(y_test_raw)
final_predicted_price = np.expm1(log_preds)

# 3. 전처리 전(Baseline Dummy) 성능 계산
# 아무 전처리도 안 했을 때의 기준점 지표 계산을 위해 원본 데이터 평균으로 예측값 가정
raw_mean_price = np.mean(raw_df["price"])
baseline_preds = np.full_like(final_actual_price, raw_mean_price)

# 최종 평가지표 연산 (RMSE over Log Price = 대회의 진짜 성적인 RMSLE)
baseline_rmsle = np.sqrt(
    mean_squared_error(np.log1p(final_actual_price), np.log1p(baseline_preds))
)
final_rmsle = np.sqrt(
    mean_squared_error(np.log1p(final_actual_price), np.log1p(final_predicted_price))
)

# 4. 정리서 요구사항에 따른 결과 대조 출력
print("\n" + "=" * 50)
print("📊 MERCARI PRICE CHALLENGE 모델 성능 검증 결과")
print("=" * 50)
print(f"❌ 고급 전처리 전 (Baseline) RMSLE 오차 점수 : {baseline_rmsle:.4f}")
print(f"✨ 고급 전처리 후 (LightGBM)  RMSLE 오차 점수 : {final_rmsle:.4f}")
print("-" * 50)
print(
    f"💡 결론: 오차가 {baseline_rmsle - final_rmsle:.4f} 만큼 대폭 감소하여 예측 성능이 획기적으로 개선되었습니다!"
)
print("=" * 50)

💡 [Hyperopt] 최적의 하이퍼파라미터 탐색을 시작합니다...
100%|██████████| 3/3 [34:50<00:00, 696.90s/trial, best loss: 0.7511631626118326]
🎯 최적화 완료된 파라미터: {'learning_rate': np.float64(0.0787149647354911), 'max_depth': 10, 'num_leaves': 31, 'subsample': np.float64(0.8876595502489376)}

📊 MERCARI PRICE CHALLENGE 모델 성능 검증 결과
❌ 고급 전처리 전 (Baseline) RMSLE 오차 점수 : 0.8205
✨ 고급 전처리 후 (LightGBM)  RMSLE 오차 점수 : 0.5325
--------------------------------------------------
💡 결론: 오차가 0.2880 만큼 대폭 감소하여 예측 성능이 획기적으로 개선되었습니다!


In [ ]:
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, fmin, hp, tpe  # STATUS_OK 임포트 추가
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer, StandardScaler  # LabelBinarizer 추가

warnings.filterwarnings("ignore")

# =====================================================================
# [단계 1] 전처리 전 기본 Baseline 평가를 위한 원본 데이터 보존
# =====================================================================
# 전처리 후 결과와 비교하기 위해 가공되지 않은 순수 원본 상태 기록
mercari_df= pd.read_csv('../mercari_train.tsv',sep='\t')
raw_df = mercari_df.copy()


# =====================================================================
# [단계 2] 고급 전처리 & 데이터 클리닝
# =====================================================================
# 1. 가격 비즈니스 로직 적용 ($3 미만 오류 데이터 제거)
mercari_df = mercari_df[mercari_df["price"] >= 3].reset_index(drop=True)

# 2. 텍스트 기본 축약어 및 정규화 매핑 사전 정의
decontract_dict = {
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "i'm": "i am",
    "it's": "it is",
}


def clean_text(text):
    if not isinstance(text, str):
        return "Missing"
    text = text.lower()
    # 축약어 풀기
    for word, replacement in decontract_dict.items():
        text = text.replace(word, replacement)
    # 특수문자 제거 (영문, 숫자, 공백만 남기기)
    text = pd.Series(text).str.replace(r"[^a-zA-Z0-9\s]", "", regex=True).values[0]
    return text


# 3. 텍스트 컬럼 전체 정제 및 결측치 기본 처리
mercari_df["name"] = mercari_df["name"].apply(clean_text)
mercari_df['item_description'] = mercari_df['item_description'].fillna('')
## 결측치를 공백으로 두니 오차가 오히려 줄었다. 
mercari_df["item_description"] = mercari_df["item_description"].apply(
    clean_text
)


# 4. 고수들의 꿀팁: 상품명(name)에서 브랜드 추출하여 brand_name 결측치 구출하기
existing_brands = set(mercari_df["brand_name"].dropna().unique())


def extract_brand(row):
    if pd.isna(row["brand_name"]):
        words = row["name"].split()
        for word in words:
            if word.title() in existing_brands:
                return word.title()  # 상품명에 존재하는 진짜 브랜드 매칭
        return "Missing"
    return row["brand_name"]


mercari_df["brand_name"] = mercari_df.apply(extract_brand, axis=1)


# =====================================================================
# [단계 3] 피쳐 구조화 (5가지 새로운 파생 변수 생성)
# =====================================================================
# 1~3. 카테고리 대/중/소분류 3개 컬럼으로 슬래시(/) 분할 (버그 수정 완료)
mercari_df["category_name"] = mercari_df["category_name"].fillna("Missing")
categories = mercari_df["category_name"].str.split("/", expand=True)

mercari_df["cat_main"] = categories[0].fillna("Missing")
mercari_df["cat_sub1"] = (
    categories[1].fillna("Missing") if categories.shape[1] > 1 else "Missing"
)
mercari_df["cat_sub2"] = (
    categories[2].fillna("Missing") if categories.shape[1] > 2 else "Missing"
)

# 4. 제품 설명(item_description)의 단어 개수 계산 (수치형)
mercari_df["desc_word_count"] = (
    mercari_df["item_description"].fillna("").apply(lambda x: len(x.split()))
)

# 5. 상품 이름(name)의 글자 길이 계산 (수치형)
mercari_df["name_len"] = mercari_df["name"].fillna("").apply(len)


# =====================================================================
# [단계 4] 인코딩, 벡터화 및 통합 특성 행렬 구축
# =====================================================================
# 1. 텍스트 벡터화 (상품명 & 설명) - 기존 유지
cnt_vec = CountVectorizer(ngram_range=(1, 2), max_features=25000)
X_name = cnt_vec.fit_transform(mercari_df["name"])

tfidf_vec = TfidfVectorizer(max_features=40000, stop_words="english")
X_desc = tfidf_vec.fit_transform(mercari_df["item_description"])

print("⏳ [업그레이드] 범주형 피처들을 LabelBinarizer 정석 원-핫 인코딩으로 변환 중...")
# 2. 범주형 인코딩 (CountVectorizer 편법 대신 정석 LabelBinarizer 적용)
lb_brand_name = LabelBinarizer(sparse_output=True)
X_brand = lb_brand_name.fit_transform(mercari_df["brand_name"])

lb_cat_main = LabelBinarizer(sparse_output=True)
X_cat_main = lb_cat_main.fit_transform(mercari_df["cat_main"])

lb_cat_sub1 = LabelBinarizer(sparse_output=True)
X_cat_sub1 = lb_cat_sub1.fit_transform(mercari_df["cat_sub1"])

lb_cat_sub2 = LabelBinarizer(sparse_output=True)
X_cat_sub2 = lb_cat_sub2.fit_transform(mercari_df["cat_sub2"])

# 3. 1글자 숫자 데이터도 에러 없이 완벽하게 변환 완료
lb_item_cond_id = LabelBinarizer(sparse_output=True)
X_condition = lb_item_cond_id.fit_transform(mercari_df["item_condition_id"])

lb_shipping = LabelBinarizer(sparse_output=True)
X_shipping = lb_shipping.fit_transform(mercari_df["shipping"])

# 4. 수치형 피처 정규화 (2차 실험 변화 부분 유지)
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(
    mercari_df[["desc_word_count", "name_len"]].values
)

# 5. 하나의 거대한 통합 특성 행렬(X)로 완성 (새로운 라벨바이너라이저 변수 매핑)
X_features_matrix = hstack(
    (
        X_name,
        X_desc,
        X_brand,
        X_cat_main,
        X_cat_sub1,
        X_cat_sub2,
        X_condition,
        X_shipping,
        X_numeric_scaled,
    )
).tocsr()


# =====================================================================
# [단계 5] 모의고사를 위한 데이터셋 분할 (Train / Test)
# =====================================================================
log_price = np.log1p(mercari_df["price"]).values.reshape(-1, 1)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X_features_matrix, log_price, test_size=0.2, random_state=42
)


# =====================================================================
# [단계 6] 스케일링 (StandardScaler 종속 변수 표준화 및 글로벌화)
# =====================================================================
global y_scalar
y_scalar = StandardScaler()

y_train = y_scalar.fit_transform(y_train_raw)
y_test = y_scalar.transform(y_test_raw)

y_train_flat = y_train.ravel()
y_test_flat = y_test.ravel()


# =====================================================================
# [단계 7] Hyperopt를 통한 LightGBM 하이퍼파라미터 최적화
# =====================================================================
space = {
    "learning_rate": hp.loguniform("learning_rate", np.log(0.05), np.log(0.2)),
    "max_depth": hp.choice("max_depth", [5, 7, 10]),
    "num_leaves": hp.choice("num_leaves", [31, 63, 127]),
    "subsample": hp.uniform("subsample", 0.7, 1.0),
}


def objective(params):
    model = lgb.LGBMRegressor(
        **params,
        n_estimators=100,  # 최적 조건 반영
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train_flat)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test_flat, preds))
    return {"loss": rmse, "status": STATUS_OK}


print("💡 [Hyperopt] 최적의 하이퍼파라미터 탐색을 시작합니다...")
best_params = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=3)

max_depth_options = [5, 7, 10]
num_leaves_options = [31, 63, 127]
best_params["max_depth"] = max_depth_options[best_params["max_depth"]]
best_params["num_leaves"] = num_leaves_options[best_params["num_leaves"]]
print(f"🎯 최적화 완료된 파라미터: {best_params}")


# =====================================================================
# [단계 8] 최적의 모델 학습 및 최종 결과 분석 (전/후 비교 출력)
# =====================================================================
best_lgb = lgb.LGBMRegressor(**best_params, n_estimators=200, n_jobs=-1, verbose=-1)
best_lgb.fit(X_train, y_train_flat)

scaled_preds = best_lgb.predict(X_test).reshape(-1, 1)
log_preds = y_scalar.inverse_transform(scaled_preds)

final_actual_price = np.expm1(y_test_raw)
final_predicted_price = np.expm1(log_preds)

raw_mean_price = np.mean(raw_df["price"])
baseline_preds = np.full_like(final_actual_price, raw_mean_price)

baseline_rmsle = np.sqrt(
    mean_squared_error(np.log1p(final_actual_price), np.log1p(baseline_preds))
)
final_rmsle = np.sqrt(
    mean_squared_error(np.log1p(final_actual_price), np.log1p(final_predicted_price))
)

print("\n" + "=" * 50)
print("📊 MERCARI PRICE CHALLENGE 모델 성능 검증 결과")
print("=" * 50)
print(f"❌ 고급 전처리 전 (Baseline) RMSLE 오차 점수 : {baseline_rmsle:.4f}")
print(f"✨ 고급 전처리 후 (LightGBM)  RMSLE 오차 점수 : {final_rmsle:.4f}")
print("=" * 50)

⏳ [업그레이드] 범주형 피처들을 LabelBinarizer 정석 원-핫 인코딩으로 변환 중...
💡 [Hyperopt] 최적의 하이퍼파라미터 탐색을 시작합니다...
100%|██████████| 3/3 [24:47<00:00, 495.84s/trial, best loss: 0.7225677817744529]
🎯 최적화 완료된 파라미터: {'learning_rate': np.float64(0.10996717067012518), 'max_depth': 10, 'num_leaves': 63, 'subsample': np.float64(0.9879887958202926)}

📊 MERCARI PRICE CHALLENGE 모델 성능 검증 결과
❌ 고급 전처리 전 (Baseline) RMSLE 오차 점수 : 0.8205
✨ 고급 전처리 후 (LightGBM)  RMSLE 오차 점수 : 0.5385
